# Đánh giá và Phân tích Dữ liệu Screen OCR

Notebook này phân tích 2 nguồn dữ liệu hiện có trong project:
- **Synthetic Data**: Ảnh được sinh tự động bằng Pillow đa luồng.
- **Real Data**: Ảnh thật được crawler thu thập từ ứng dụng Windows.

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('')))

import json
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Đảm bảo hiển thị đẹp
plt.rcParams['figure.figsize'] = (10, 6)
plt.style.use('ggplot')

## 1. Load Data

In [ ]:
def load_synthetic_data(dir_path):
    p = Path(dir_path)
    labels_file = p / "labels.jsonl"
    samples = []
    if not labels_file.exists():
        return samples
    
    with open(labels_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            img_path = p / data['file']
            samples.append({
                'source': 'synthetic',
                'img_path': str(img_path),
                'label': data['label'],
                'control_type': 'SyntheticText'
            })
    return samples

def load_real_data(dir_path):
    p = Path(dir_path)
    samples = []
    if not p.exists():
        return samples
        
    for json_path in p.glob("*.json"):
        data = json.loads(json_path.read_text(encoding='utf-8'))
        img_path = p / f"{json_path.stem}.png"
        samples.append({
            'source': 'real',
            'img_path': str(img_path),
            'label': data['label'],
            'control_type': data.get('control_type', 'Unknown')
        })
    return samples

synth_train = load_synthetic_data('../data/synthetic/train')
synth_val = load_synthetic_data('../data/synthetic/val')
real_data = load_real_data('../data/real')

all_data = synth_train + synth_val + real_data
df = pd.DataFrame(all_data)
df['label_len'] = df['label'].apply(len)

print(f"Tổng số mẫu: {len(df):,}")
print(f"- Synthetic: {len(synth_train) + len(synth_val):,}")
print(f"- Real: {len(real_data):,}")

## 2. Thống kê độ dài chuỗi (Label Length Distribution)

In [ ]:
df.groupby('source')['label_len'].hist(alpha=0.5, bins=20, legend=True)
plt.title('Phân bố độ dài chuỗi (Label Length)')
plt.xlabel('Số ký tự')
plt.ylabel('Tần suất')
plt.show()

print("Chiều dài tối đa:", df['label_len'].max())
print("Chiều dài trung bình:", round(df['label_len'].mean(), 2))

## 3. Phân tích nhiễu (Control Types in Real Data)

In [ ]:
real_df = df[df['source'] == 'real']
if not real_df.empty:
    print("Tỉ lệ các loại Control trong dữ liệu thực tế:")
    print(real_df['control_type'].value_counts())
else:
    print("Chưa có dữ liệu real.")

## 4. Trực quan hoá ảnh (Visualizing Samples)

In [ ]:
def show_samples(dataframe, n=6, title="Samples"):
    if dataframe.empty:
        return
    samples = dataframe.sample(min(n, len(dataframe)))
    
    fig, axes = plt.subplots(len(samples), 1, figsize=(6, 2 * len(samples)))
    if len(samples) == 1:
        axes = [axes]
        
    for ax, (_, row) in zip(axes, samples.iterrows()):
        img = cv2.imread(row['img_path'])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        ax.set_title(f"Label: {row['label']} | Type: {row['control_type']}")
        ax.axis('off')
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

show_samples(df[df['source'] == 'synthetic'], n=5, title="Synthetic Data")
show_samples(df[df['source'] == 'real'], n=5, title="Real Data")